In [9]:
import sys, platform
ON_LINUXONE = platform.machine() == "s390x"
DATA_PATH = ("/home/ibm-user/transactions_history.csv" if ON_LINUXONE
             else "transactions_history.csv")
SEED = 42
TRAIN_END_YEAR = 2015         # train: up to 2015
VALID_YEARS = (2016, 2017)    # validation: compare models here
TEST_YEARS = (2018, 2019)     # test: use once in step 22

import pandas as pd
import numpy as np
import sklearn
import xgboost
import tensorflow as tf
import keras

print("Machine:", platform.machine(), "| Python:", sys.version.split()[0])

Machine: s390x | Python: 3.12.15


In [10]:
# 1. Load data
df = pd.read_csv(DATA_PATH)

# 2. Clean Amount: "$134.09" -> 134.09
df['Amount_num'] = df['Amount'].astype(str).str.replace('$', '', regex=False).astype(float)

# 3. Clean 'Is Fraud?' -> 1 / 0
# Bypass the object dtype check to support Pandas 3.x StringDtype
df['is_fraud'] = (df['Is Fraud?'].astype(str).str.strip().str.lower() == 'yes').astype(int)

# 4. Create single timestamp 'ts'
time_split = df['Time'].astype(str).str.split(':', expand=True)
df['hour'] = time_split[0].astype(int)
df['minute'] = time_split[1].astype(int)

df['ts'] = pd.to_datetime(dict(
    year=df['Year'],
    month=df['Month'],
    day=df['Day'],
    hour=df['hour'],
    minute=df['minute']
))

# 5. Sort chronologically
df = df.sort_values(by=['User', 'Card', 'ts']).reset_index(drop=True)

# 6. Assign train / valid / test splits by year
def assign_split(year):
    if year <= TRAIN_END_YEAR:
        return 'train'
    elif year in VALID_YEARS:
        return 'valid'
    elif year in TEST_YEARS:
        return 'test'
    else:
        return 'drop'

df['split'] = df['Year'].apply(assign_split)

# 7. Drop 2020+
df = df[df['split'] != 'drop'].reset_index(drop=True)

# 8. Print fraud counts to verify
fraud_counts = df.groupby('split')['is_fraud'].sum()
print("--- Fraud Count per Split ---")
print(fraud_counts)

--- Fraud Count per Split ---
split
test      60
train    178
valid     33
Name: is_fraud, dtype: int64


In [7]:
from sklearn.metrics import precision_score, recall_score, precision_recall_curve, auc, confusion_matrix
import numpy as np

def report(y_true, scores, threshold=0.5):
    preds = (scores >= threshold).astype(int)
    precision = precision_score(y_true, preds, zero_division=0)
    recall = recall_score(y_true, preds, zero_division=0)
    
    precision_curve, recall_curve, _ = precision_recall_curve(y_true, scores)
    pr_auc = auc(recall_curve, precision_curve)
    
    # FIX: Added labels=[0,1] to handle edge cases where a batch has no fraud
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    alerts_per_1000 = ((tp + fp) / len(y_true)) * 1000
    
    print(f"Precision:           {precision:.4f}")
    print(f"Recall:              {recall:.4f}")
    print(f"PR-AUC:              {pr_auc:.4f}")
    print(f"False Positive Rate: {fpr:.6f}")
    print(f"Alerts per 1,000:    {alerts_per_1000:.2f}")
    
    return precision, recall, pr_auc, fpr, alerts_per_1000

# Test
np.random.seed(SEED)
_ = report(np.random.randint(0, 2, 10000), np.random.rand(10000))

Precision:           0.5040
Recall:              0.4965
PR-AUC:              0.4986
False Positive Rate: 0.486136
Alerts per 1,000:    491.30


In [11]:
# 1. Base numerical and time features
df['amount'] = df['Amount_num']
df['weekday'] = df['ts'].dt.dayofweek
df['is_refund'] = (df['amount'] < 0).astype(int)

# 2. Chip / Swipe / Online (One-hot encode)
chip_dummies = pd.get_dummies(df['Use Chip'], prefix='tx_type', dtype=int)
df = pd.concat([df, chip_dummies], axis=1)

# 3. Foreign Merchant (US states are 2 letters; everything else is foreign)
df['foreign_merchant'] = (~df['Merchant State'].astype(str).str.match(r'^[A-Za-z]{2}$', na=False)).astype(int)

# 4. Errors (One-hot encode comma-separated error types)
df['Errors?'] = df['Errors?'].fillna('')
errors_dummies = df['Errors?'].str.get_dummies(sep=',').add_prefix('err_')
df = pd.concat([df, errors_dummies], axis=1)

# 5. Define the final feature list
FEATURE_COLUMNS = ['amount', 'hour', 'weekday', 'is_refund', 'foreign_merchant', 'MCC'] + \
                  list(chip_dummies.columns) + list(errors_dummies.columns)

print("--- Feature Columns ---")
for col in FEATURE_COLUMNS:
    print(col)

print("\nFeature table shape:", df[FEATURE_COLUMNS].shape)

--- Feature Columns ---
amount
hour
weekday
is_refund
foreign_merchant
MCC
tx_type_Chip Transaction
tx_type_Online Transaction
tx_type_Swipe Transaction
err_Bad CVV
err_Bad Card Number
err_Bad Expiration
err_Bad PIN
err_Bad Zipcode
err_Insufficient Balance
err_Technical Glitch

Feature table shape: (242493, 16)


In [12]:
# FIX: Added 'amount' to tie-break simultaneous timestamps
df = df.sort_values(by=['User', 'Card', 'ts', 'amount']).reset_index(drop=True)

df_time = df.set_index('ts')
grouped_time = df_time.groupby(['User', 'Card'])

# 1. Velocity (FIX: Subtract 1 from counts, subtract current amount from sums)
df['count_1h'] = grouped_time['amount'].transform(lambda x: x.rolling('1h').count()).reset_index(drop=True) - 1
df['count_24h'] = grouped_time['amount'].transform(lambda x: x.rolling('24h').count()).reset_index(drop=True) - 1
df['count_7d'] = grouped_time['amount'].transform(lambda x: x.rolling('7D').count()).reset_index(drop=True) - 1 # Fixed 7D warning
df['amt_24h'] = grouped_time['amount'].transform(lambda x: x.rolling('24h').sum()).reset_index(drop=True) - df['amount']

# 2. Minutes since previous
df['mins_since_last'] = df.groupby(['User', 'Card'])['ts'].diff().dt.total_seconds() / 60.0
df['mins_since_last'] = df['mins_since_last'].fillna(0)

# 3. Behavior (FIX: Shift median backward by 1 so it only looks at the past)
df['median_so_far'] = df.groupby(['User', 'Card'])['amount'].transform(lambda x: x.shift().expanding().median())
df['median_so_far'] = df['median_so_far'].fillna(0)
df['ratio_median'] = np.where(df['median_so_far'] > 0, df['amount'] / df['median_so_far'], 0)

df['first_time_merchant'] = (~df.duplicated(subset=['User', 'Card', 'Merchant Name'])).astype(int)
df['first_time_city'] = (~df.duplicated(subset=['User', 'Card', 'Merchant City'])).astype(int)

# Update feature list
NEW_FEATURES = ['count_1h', 'count_24h', 'count_7d', 'amt_24h', 'mins_since_last', 
                'ratio_median', 'first_time_merchant', 'first_time_city']
FEATURE_COLUMNS.extend([f for f in NEW_FEATURES if f not in FEATURE_COLUMNS])

print("--- Running Strict Leakage Check ---")
card_df = df[(df['User'] == df['User'].iloc[0]) & (df['Card'] == df['Card'].iloc[0])].copy()
half_idx = len(card_df) // 2

# Check if first half median matches the full calculation perfectly
first_half_median = card_df.iloc[:half_idx]['amount'].shift().expanding().median().fillna(0)
diff = (first_half_median - card_df['median_so_far'].iloc[:half_idx]).abs().max()

if diff == 0:
    print("Leakage check PASSED! (Rolling features are strictly backward-looking)")
else:
    print(f"Leakage check FAILED! Difference: {diff}")

--- Running Strict Leakage Check ---
Leakage check PASSED! (Rolling features are strictly backward-looking)


In [13]:
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow.keras as keras
import tensorflow as tf

# Set random seed for reproducibility
keras.utils.set_random_seed(SEED)

# 1. Isolate the training and validation sets
train_df = df[df['split'] == 'train'].copy()
valid_df = df[df['split'] == 'valid'].copy()

X_train = train_df[FEATURE_COLUMNS].copy()
y_train = train_df['is_fraud']

X_valid = valid_df[FEATURE_COLUMNS].copy()
y_valid = valid_df['is_fraud']

# Calculate class weights for Keras
num_non_fraud = (y_train == 0).sum()
num_fraud = (y_train == 1).sum()
total = num_non_fraud + num_fraud

weight_for_0 = (1 / num_non_fraud) * (total / 2.0)
weight_for_1 = (1 / num_fraud) * (total / 2.0)
class_weight = {0: weight_for_0, 1: weight_for_1}

# 2. Scale numerical columns (Neural networks require scaled data)
scaler = StandardScaler()

# Scale train data
X_train_scaled = scaler.fit_transform(X_train)
# Scale validation data using the scaler fit on training data
X_valid_scaled = scaler.transform(X_valid)

# 3. Build the Keras Dense Model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid') # Binary classification output
])

# Compile the model optimizing for PR-AUC
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(curve='PR', name='pr_auc')]
)

# 4. Define Early Stopping (stops training if validation PR-AUC stops improving)
early_stopping = EarlyStopping(
    monitor='val_pr_auc', 
    patience=5, 
    mode='max', 
    restore_best_weights=True
)

# 5. Train the TensorFlow model
print("\nTraining TensorFlow model...")
history = model.fit(
    X_train_scaled, 
    y_train,
    epochs=30,
    batch_size=2048,
    validation_data=(X_valid_scaled, y_valid),
    class_weight=class_weight,
    callbacks=[early_stopping],
    verbose=1
)

# 6. Predict on the validation set and evaluate
valid_scores_tf = model.predict(X_valid_scaled).ravel()

print("\n--- TensorFlow Validation Results ---")
tf_metrics = report(y_valid, valid_scores_tf, threshold=0.5)

/opt/venv/lib64/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Training TensorFlow model...
Epoch 1/30


2026-10-17 10:07:40.486419: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 16828416 exceeds 10% of free system memory.


86/86 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.5645 - pr_auc: 0.0113 - val_loss: 0.4091 - val_pr_auc: 0.0090
Epoch 2/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.3517 - pr_auc: 0.0202 - val_loss: 0.4091 - val_pr_auc: 0.0090
Epoch 3/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.2827 - pr_auc: 0.0335 - val_loss: 0.4091 - val_pr_auc: 0.0090
Epoch 4/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.2808 - pr_auc: 0.0351 - val_loss: 0.4091 - val_pr_auc: 0.0090
Epoch 5/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.2889 - pr_auc: 0.0319 - val_loss: 0.4091 - val_pr_auc: 0.0090
Epoch 6/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.2550 - pr_auc: 0.0378 - val_loss: 0.4091 - val_pr_auc: 0.0090
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 0s 348us/step

--- TensorFlow Validation Results ---
Precision:           0.0035
Recall:              0.8182
PR-AUC:              0.0090
False Positive Rate: 0.230957
Alerts per 1,000:    231.53


In [14]:
from tensorflow.keras.layers import LSTM, Masking
import numpy as np

SEQ_LEN = 10

def create_sequences(split_name):
    split_data = df[df['split'] == split_name].copy()
    
    # Scale features using the scaler from Step 10
    scaled_features = scaler.transform(split_data[FEATURE_COLUMNS])
    
    # Reattach identifiers for grouping
    split_data_scaled = pd.DataFrame(scaled_features, columns=FEATURE_COLUMNS, index=split_data.index)
    split_data_scaled['User'] = split_data['User']
    split_data_scaled['Card'] = split_data['Card']
    split_data_scaled['is_fraud'] = split_data['is_fraud']
    
    X_seq, y_seq = [], []
    
    # Build rolling windows of up to 10 past transactions per card
    for _, group in split_data_scaled.groupby(['User', 'Card']):
        features = group[FEATURE_COLUMNS].values
        labels = group['is_fraud'].values
        
        for i in range(len(group)):
            start_idx = max(0, i - SEQ_LEN + 1)
            seq = features[start_idx : i + 1]
            
            # Pad with zeros if the card has fewer than 10 transactions so far
            if len(seq) < SEQ_LEN:
                pad = np.zeros((SEQ_LEN - len(seq), len(FEATURE_COLUMNS)))
                seq = np.vstack([pad, seq])
                
            X_seq.append(seq)
            y_seq.append(labels[i])
            
    return np.array(X_seq), np.array(y_seq)

print("Building sequences for training and validation... (This may take a minute)")
X_train_seq, y_train_seq = create_sequences('train')
X_valid_seq, y_valid_seq = create_sequences('valid')

# Build the LSTM Model
lstm_model = Sequential([
    Masking(mask_value=0., input_shape=(SEQ_LEN, len(FEATURE_COLUMNS))),
    LSTM(32, activation='tanh', return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(curve='PR', name='pr_auc')]
)

# Train the LSTM model
print("\nTraining LSTM model...")
lstm_model.fit(
    X_train_seq, y_train_seq,
    epochs=15,
    batch_size=1024,
    validation_data=(X_valid_seq, y_valid_seq),
    class_weight=class_weight,
    callbacks=[EarlyStopping(monitor='val_pr_auc', patience=4, mode='max', restore_best_weights=True)],
    verbose=1
)

# Predict and evaluate
valid_scores_lstm = lstm_model.predict(X_valid_seq).ravel()

print("\n--- LSTM Sequence Validation Results ---")
lstm_metrics = report(y_valid_seq, valid_scores_lstm, threshold=0.5)

Building sequences for training and validation... (This may take a minute)


/opt/venv/lib64/python3.12/site-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Training LSTM model...
Epoch 1/15


2026-10-17 10:12:52.957820: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 168284160 exceeds 10% of free system memory.


170/172 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.5736 - pr_auc: 0.0762

2026-10-17 10:12:57.270829: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 32307840 exceeds 10% of free system memory.


172/172 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - loss: 0.4581 - pr_auc: 0.1675 - val_loss: 0.3481 - val_pr_auc: 0.0350
Epoch 2/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 0.2192 - pr_auc: 0.3734 - val_loss: 0.2346 - val_pr_auc: 0.0637
Epoch 3/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 0.1458 - pr_auc: 0.4037 - val_loss: 0.2118 - val_pr_auc: 0.0603
Epoch 4/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.1014 - pr_auc: 0.4177 - val_loss: 0.2052 - val_pr_auc: 0.0667
Epoch 5/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0798 - pr_auc: 0.3480 - val_loss: 0.1864 - val_pr_auc: 0.0595
Epoch 6/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 0.0664 - pr_auc: 0.3920 - val_loss: 0.2667 - val_pr_auc: 0.0486
Epoch 7/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0469 - pr_auc: 0.4343 - val_loss: 0.1550 - val_pr_auc: 0.0637
Epoch 8/15
172/172 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.0381 - pr_auc: 0.3769 - val_loss: 0.1294 - val_pr_auc: 0.0557
1052/1052 ━